In [1]:
import json
from pathlib import Path
import pandas as pd

# Load cleaned json file from data/interim
data_path = Path("../data/interim/cleaned_details.json")

records = json.loads(data_path.read_text(encoding="utf-8"))
print(f"Total records: {len(records)}")

Total records: 24092


In [2]:
df = pd.DataFrame([
    {
        "title":            r.get("title"),
        "series_name":      r.get("series_name"),
        "episode_number":   r.get("episode_number"),
        "description":      r.get("description"),
        "duration_minutes": r.get("duration_minutes"),
        "release_date":     r.get("release_date"),
        "label":            r.get("label"),
        "cover_url":        r.get("cover_url"),
        "order_number":     r.get("order_number"),
        "source_url":       r.get("source_url"),
        "n_speakers":       len(r.get("speakers") or []),
        "n_genres":         len(r.get("genres") or []),
    }
    for r in records
])

print(f"Shape: {df.shape}")

Shape: (24092, 12)


In [3]:
with_description = sum(1 for r in records if r.get("description"))
print(f"With description:    {with_description}")
print(f"Without description: {len(records) - with_description}")
print(f"Percentage:          {with_description / len(records) * 100:.1f}%")
df["description"].notna().sum()

With description:    12899
Without description: 11193
Percentage:          53.5%


np.int64(12899)

In [4]:
# Data types and memory usage
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24092 entries, 0 to 24091
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             24092 non-null  str    
 1   series_name       24092 non-null  str    
 2   episode_number    21497 non-null  float64
 3   description       12899 non-null  str    
 4   duration_minutes  5813 non-null   float64
 5   release_date      10147 non-null  str    
 6   label             12906 non-null  str    
 7   cover_url         10894 non-null  str    
 8   order_number      7141 non-null   str    
 9   source_url        12938 non-null  str    
 10  n_speakers        24092 non-null  int64  
 11  n_genres          24092 non-null  int64  
dtypes: float64(2), int64(2), str(8)
memory usage: 2.2 MB


In [5]:
df.head()

,title,series_name,episode_number,description,duration_minutes,release_date,label,cover_url,order_number,source_url,n_speakers,n_genres
0,Serie,Serien: Beyond the Veil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,Nr,Serien: Beyond the Veil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,Seancé,Beyond the Veil,1.0,"Es gibt Menschen unter uns, die, wenn sie ihre...",NaN,15.10.2010,Maritim,https://www.hoerspiele.de/bilder/bilder/_b23/b...,NaN,https://www.hoerspiele.de/hsp_anzeige.asp?code...,14,2
3,Tabu,Beyond the Veil,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,Frevel,Beyond the Veil,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [6]:
# Missing values — how complete is the data?
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing": missing, "percent": missing_pct})

,missing,percent
duration_minutes,18279,75.9
order_number,16951,70.4
release_date,13945,57.9
cover_url,13198,54.8
description,11193,46.5
label,11186,46.4
source_url,11154,46.3
episode_number,2595,10.8
title,0,0.0
series_name,0,0.0


In [7]:
# Episodes without a title — should be 0, otherwise parser issue
no_title = df[df["title"].isnull()]
print(f"Episodes without title: {len(no_title)}")
if len(no_title) > 0:
    print(no_title[["source_url"]].head(10))

Episodes without title: 0


In [8]:
# Series: count and episodes per series
series_counts = df.groupby("series_name").size().sort_values(ascending=False)
print(f"Number of series: {len(series_counts)}\n")
series_counts.head(20)

Number of series: 2558



series_name
(Diverse)                     791
Märchen                       410
Die Drei ???                  405
TKKG                          323
John Sinclair                 317
Lieder                        241
Gruselkabinett                207
Perry Rhodan (Silberbände)    205
Benjamin Blümchen             205
Bibi Blocksberg               204
Filme                         188
Fünf Freunde                  184
Sherlock Holmes Chronicles    164
Walt Disney                   159
Geister-Schocker              150
Die Drei ??? Kids             144
Pater Brown                   144
Karl May                      143
Die drei !!!                  141
Teufelskicker                 135
dtype: int64

In [9]:
# Parse release date and check for problems
df["release_date_parsed"] = pd.to_datetime(
    df["release_date"], format="%d.%m.%Y", errors="coerce"
)

n_missing = df["release_date"].isnull().sum()
n_failed  = df["release_date_parsed"].isnull().sum()
n_errors  = n_failed - n_missing

print(f"No date at all:       {n_missing}")
print(f"Failed to parse:      {n_errors}  ← cleaning candidates")
print(f"Successfully parsed:  {len(df) - n_failed}")

# Which values failed?
df[df["release_date"].notnull() & df["release_date_parsed"].isnull()][
    ["title", "release_date", "source_url"]
].head(20)

No date at all:       13945
Failed to parse:      0  ← cleaning candidates
Successfully parsed:  10147


,title,release_date,source_url


In [10]:
# Releases per year
df["year"] = df["release_date_parsed"].dt.year
df["year"].value_counts().sort_index()

year
1089.0      2
1910.0      1
1912.0      1
1938.0      1
1956.0      1
1960.0      2
1962.0      1
1963.0      1
1969.0      2
1976.0      3
1979.0      9
1980.0     12
1981.0      7
1982.0      2
1983.0      4
1984.0      2
1985.0      2
1986.0      3
1987.0      2
1988.0      8
1989.0      2
1990.0      1
1991.0      5
1992.0     10
1993.0     10
1994.0      9
1995.0     13
1996.0     13
1997.0     23
1998.0     35
1999.0     39
2000.0     66
2001.0    254
2002.0    285
2003.0    250
2004.0    373
2005.0    431
2006.0    615
2007.0    697
2008.0    686
2009.0    663
2010.0    615
2011.0    498
2012.0    400
2013.0    335
2014.0    254
2015.0    293
2016.0    278
2017.0    282
2018.0    244
2019.0    226
2020.0    317
2021.0    427
2022.0    326
2023.0    349
2024.0    318
2025.0    313
2026.0    124
2027.0      1
2028.0      1
Name: count, dtype: int64

In [11]:
# Outliers in duration
print(df["duration_minutes"].describe())

print("\nVery short (< 5 min):")
print(df[df["duration_minutes"] < 5][["title", "series_name", "duration_minutes"]].head(10))

print("\nVery long (> 180 min):")
print(df[df["duration_minutes"] > 180][["title", "series_name", "duration_minutes"]].head(10))

count    5813.000000
mean       58.881108
std        64.409094
min         2.100000
25%        47.250000
50%        58.260000
75%        68.530000
max      4720.000000
Name: duration_minutes, dtype: float64

Very short (< 5 min):
                                     title                     series_name  \
2074                           6. Dezember  Die Drei ??? und der 5. Advent   
2082                          14. Dezember  Die Drei ??? und der 5. Advent   
2084                          16. Dezember  Die Drei ??? und der 5. Advent   
2086                          18. Dezember  Die Drei ??? und der 5. Advent   
2088                          20. Dezember  Die Drei ??? und der 5. Advent   
2089                          21. Dezember  Die Drei ??? und der 5. Advent   
7339                Folge 1 - Die Songs EP                     Eazy Eizbär   
7411     Die gefährliche Spur (Käse-Pizza)              Die Drei ??? Pizza   
7412   Der feuerrote Teufel (Salami-Pizza)              Die Drei ???

In [12]:
# Explore genres and count
genres_series = pd.Series([
    g for r in records for g in (r.get("genres") or [])
])
print(f"Unique genres: {genres_series.nunique()}\n")
genres_series.value_counts().head(20)

Unique genres: 20



Krimi und Detektiv         1145
Grusel und Horror          1060
Abenteuer                   949
Kinder und Jugend           844
Mystery                     586
Science Fiction             430
Action                      338
Fantasy                     226
Comedy                      168
OTV                         149
Für Mädchen                 117
Wissen und Infotainment      96
Märchen                      91
Liebe und Romantik           91
Thriller                     65
Western                      63
Musik und Lieder             61
Nur ab 18                    40
Pferdegeschichten            30
Für Jungs                    25
Name: count, dtype: int64

In [13]:
# Most active voice actors
all_speakers = pd.Series([
    s["speaker"] for r in records for s in (r.get("speakers") or [])
])
print(f"Total speaker credits: {len(all_speakers)}")
print(f"Unique speakers:       {all_speakers.nunique()}\n")
all_speakers.value_counts().head(20)

Total speaker credits: 92461
Unique speakers:       10207



Rohrbeck, Oliver          587
Mackensy, Lutz            554
Fröhlich, Andreas         476
Wawrczeck, Jens           412
von der Meden, Andreas    408
Draeger, Sascha           366
Schülke, Achim            340
Paetsch, Hans             321
Missler, Robert           305
Karallus, Thomas          305
Thormann, Jürgen          303
Lubowski, Manou           303
Schneider, Reinhilt       300
Krauss, Helmut            296
Bierstedt, Detlef         295
Bach, Patrick             292
Halver, Konrad            292
Rode, Christian           286
Draeger, Kerstin          283
Schmidt-Foß, Gerrit       272
Name: count, dtype: int64

In [14]:
# Normalize speaker names for comparison
# Format is "Lastname, Firstname" — let's check for umlaut variants
speakers_df = pd.DataFrame([
    {
        "speaker": s["speaker"],
        "role":    s["role"],
        "title":   r.get("title"),
        "series":  r.get("series_name"),
    }
    for r in records
    for s in (r.get("speakers") or [])
])

print(f"Total speaker credits: {len(speakers_df)}")
print(f"Unique speaker names:  {speakers_df['speaker'].nunique()}")
speakers_df.head(5)

Total speaker credits: 92461
Unique speaker names:  10207


,speaker,role,title,series
0,"Riedel, Lutz",Erzähler,Seancé,Beyond the Veil
1,"Kluckert, Jürgen",McAlpin (alt),Seancé,Beyond the Veil
2,"Berenz, Johannes",McAlpin (jung),Seancé,Beyond the Veil
3,"Fröhlich, Andreas",Stuart,Seancé,Beyond the Veil
4,"Schenk, Udo",Alastor,Seancé,Beyond the Veil


In [15]:
# Look for potential umlaut variants of the same person
# Strategy: normalize umlauts and compare against original
umlaut_map = str.maketrans({
    "ä": "ae", "ö": "oe", "ü": "ue",
    "Ä": "Ae", "Ö": "Oe", "Ü": "Ue",
    "ß": "ss",
})

def normalize_name(name: str) -> str:
    return name.translate(umlaut_map).lower().strip()

speakers_df["speaker_normalized"] = speakers_df["speaker"].map(normalize_name)

# Group by normalized name — if a normalized name has more than one
# original spelling, that's a candidate for deduplication
variants = (
    speakers_df.groupby("speaker_normalized")["speaker"]
    .nunique()
    .reset_index()
    .rename(columns={"speaker": "n_variants"})
)

suspicious = variants[variants["n_variants"] > 1].sort_values("n_variants", ascending=False)
print(f"Normalized names with multiple spellings: {len(suspicious)}")
suspicious.head(20)

Normalized names with multiple spellings: 1


,speaker_normalized,n_variants
5480,"maertens, michael",2


In [16]:
# Inspect the actual variants for each suspicious name
for norm_name in suspicious["speaker_normalized"].head(20):
    original_variants = (
        speakers_df[speakers_df["speaker_normalized"] == norm_name]["speaker"]
        .value_counts()
    )
    print(f"\n{norm_name}:")
    print(original_variants.to_string())


maertens, michael:
speaker
Maertens, Michael    7
Märtens, Michael     2


In [17]:
# Check for other common inconsistencies:
# trailing/leading whitespace, double spaces, etc.
has_whitespace_issue = speakers_df["speaker"].str.contains(r"^\s|\s$|\s{2,}", regex=True)
print(f"Names with whitespace issues: {has_whitespace_issue.sum()}")
speakers_df[has_whitespace_issue][["speaker"]].drop_duplicates().head(20)

Names with whitespace issues: 0


,speaker


In [18]:
# Same check for roles
roles_df = speakers_df["role"].value_counts()
print(f"Unique roles: {len(roles_df)}\n")

# Roles that look like they might be the same
# e.g. "Erzähler" vs "Erzaehler" vs "erzähler"
roles_normalized = pd.Series(
    speakers_df["role"].map(normalize_name).value_counts()
)

role_variants = (
    speakers_df.groupby(speakers_df["role"].map(normalize_name))["role"]
    .nunique()
    .reset_index(name="n_variants")  # direkt den neuen Spaltennamen setzen
)

suspicious_roles = role_variants[role_variants["n_variants"] > 1]
print(f"Roles with multiple spellings: {len(suspicious_roles)}")
suspicious_roles.head(20)

Unique roles: 33676

Roles with multiple spellings: 59


,role,n_variants
623,aeltere frau,2
626,aelterer polizist,2
922,aleksa meisner,2
1014,alf,2
1193,alte dame,2
1196,alte frau,2
1221,alter mann,2
2097,ascari da vivo,2
2542,barbara blocksberg,2
4176,caesar,2


In [19]:
# Inspect the actual variants for each suspicious role
for norm_role in suspicious_roles["role"].head(20):
    original_variants = (
        speakers_df[speakers_df["role"].map(normalize_name) == norm_role]["role"]
        .value_counts()
    )
    print(f"\n{norm_role}:")
    print(original_variants.to_string())


aeltere frau:
role
Ältere Frau    2
ältere Frau    1

aelterer polizist:
role
älterer Polizist    1
Älterer Polizist    1

aleksa meisner:
role
Aleksa Meisner    23
aleksa Meisner     1

alf:
role
Alf    16
ALF     5

alte dame:
role
Alte Dame    4
alte Dame    1

alte frau:
role
Alte Frau    22
alte Frau     4

alter mann:
role
Alter Mann    22
alter Mann     1

ascari da vivo:
role
Ascari da Vivo    6
Ascari Da Vivo    1

barbara blocksberg:
role
Barbara Blocksberg    100
Barbara blocksberg      1

caesar:
role
Cäsar     6
Caesar    5

christopher van helsing:
role
Christopher Van Helsing    12
Christopher van Helsing     8

cora:
role
Cora    9
CORA    2

das haessliche junge entlein:
role
Das häßliche junge Entlein    1
das häßliche junge Entlein    1

delroy:
role
Delroy    1
DelRoy    1

der alte mann:
role
Der alte Mann    2
der alte Mann    1

der grosse wolf:
role
Der große Wolf    2
Der Große Wolf    1

der inspektor:
role
Der Inspektor    2
der Inspektor    1

der schwarze 

In [20]:
# How often does each role appear? Helps spot noise vs real roles
# e.g. one-off typos will have very low counts
role_counts = speakers_df["role"].value_counts()
print("Most common roles:")
print(role_counts.head(20))
print("\nRoles that appear only once (potential noise):")
print(role_counts[role_counts == 1].head(20))

Most common roles:
role
''Unbekannt''            5223
Erzähler                 3928
Polizist                  392
Sprecher des Hörbuchs     392
Erzählerin                338
Bob Andrews               335
Justus Jonas              326
Peter Shaw                325
Credits                   302
Teil der Menge            276
George                    218
Anne                      212
Mutter                    205
Julian                    194
Dick                      192
Mann                      190
Sherlock Holmes           178
Kommissar Glockner        178
Max                       165
Frau                      164
Name: count, dtype: int64

Roles that appear only once (potential noise):
role
McAlpin (alt)               1
McAlpin (jung)              1
Alastor                     1
Ethan Balfour               1
Sarah Balfour               1
Robert Balfour              1
Foggerty                    1
Georgina, genannte Georg    1
Frau Stock, Köchin          1
Herr Stock, Seemann        

In [21]:
# Summary: what needs fixing before writing to DB?
print("=== Data Quality Summary ===\n")
print(f"Speaker name variants (umlaut issues): {len(suspicious)}")
print(f"Whitespace issues in names:            {has_whitespace_issue.sum()}")
print(f"Role spelling variants:                {len(suspicious_roles)}")
print(f"\nRecords with no speakers at all:       {(df['n_speakers'] == 0).sum()}")
print(f"Records with no genres:                {(df['n_genres'] == 0).sum()}")
print(f"Records with no release date:          {df['release_date'].isnull().sum()}")
print(f"Records with no description:           {df['description'].isnull().sum()}")

=== Data Quality Summary ===

Speaker name variants (umlaut issues): 1
Whitespace issues in names:            0
Role spelling variants:                59

Records with no speakers at all:       16905
Records with no genres:                19286
Records with no release date:          13945
Records with no description:           11193
